# Ford GoBike — Phase 2: Data Cleaning
Dataset: FordGoBike.csv — 183,417 trips, Feb. 2019

In [28]:
import pandas as pd

df = pd.read_csv('FordGoBike.csv')
print(df.shape)
df.head(10)

(183416, 16)


,duration_sec,start_time,end_time,start_station_id,start_station_name,start_station_latitude,start_station_longitude,end_station_id,end_station_name,end_station_latitude,end_station_longitude,bike_id,user_type,member_birth_year,member_gender,bike_share_for_all_trip
0,52185,32:10.1,01:56.0,21.0,Montgomery St BART Station (Market St at 2nd St),37.789625,-122.400811,13.0,Commercial St at Montgomery St,37.794231,-122.402923,4902,Customer,1984.0,Male,No
1,42521,53:21.8,42:03.1,23.0,The Embarcadero at Steuart St,37.791464,-122.391034,81.0,Berry St at 4th St,37.775880,-122.393170,2535,Customer,NaN,NaN,No
2,61854,13:13.2,24:08.1,86.0,Market St at Dolores St,37.769305,-122.426826,3.0,Powell St BART Station (Market St at 4th St),37.786375,-122.404904,5905,Customer,1972.0,Male,No
3,36490,54:26.0,02:36.8,375.0,Grove St at Masonic Ave,37.774836,-122.446546,70.0,Central Ave at Fell St,37.773311,-122.444293,6638,Subscriber,1989.0,Other,No
4,1585,54:18.5,20:44.1,7.0,Frank H Ogawa Plaza,37.804562,-122.271738,222.0,10th Ave at E 15th St,37.792714,-122.248780,4898,Subscriber,1974.0,Male,Yes
5,1793,49:58.6,19:51.8,93.0,4th St at Mission Bay Blvd S,37.770407,-122.391198,323.0,Broadway at Kearny,37.798014,-122.405950,5200,Subscriber,1959.0,Male,No
6,1147,55:35.1,14:42.6,300.0,Palm St at Willow St,37.317298,-121.884995,312.0,San Jose Diridon Station,37.329732,-121.901782,3803,Subscriber,1983.0,Female,No
7,1615,41:06.8,08:02.8,10.0,Washington St at Kearny St,37.795393,-122.404770,127.0,Valencia St at 21st St,37.756708,-122.421025,6329,Subscriber,1989.0,Male,No
8,1570,41:48.8,07:59.7,10.0,Washington St at Kearny St,37.795393,-122.404770,127.0,Valencia St at 21st St,37.756708,-122.421025,6548,Subscriber,1988.0,Other,No
9,1049,49:47.7,07:17.0,19.0,Post St at Kearny St,37.788975,-122.403452,121.0,Mission Playground,37.759210,-122.421339,6488,Subscriber,1992.0,Male,No



## Data Quality Note: start_time / end_time are corrupted
These columns only contain MM:SS.s (no date/hour). Confirmed via dtype check.
Not usable for any date-based work — flagged to the team.

In [29]:
df[['duration_sec', 'start_time', 'end_time']].head(10)


,duration_sec,start_time,end_time
0,52185,32:10.1,01:56.0
1,42521,53:21.8,42:03.1
2,61854,13:13.2,24:08.1
3,36490,54:26.0,02:36.8
4,1585,54:18.5,20:44.1
5,1793,49:58.6,19:51.8
6,1147,55:35.1,14:42.6
7,1615,41:06.8,08:02.8
8,1570,41:48.8,07:59.7
9,1049,49:47.7,07:17.0


In [30]:
df['start_time'].dtype

<StringDtype(storage='python', na_value=nan)>

## Step 1: Check missing values

In [31]:
missing = df.isnull().sum()
missing_percentage = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_percentage': missing_percentage})[missing >0]

,missing_count,missing_percentage
start_station_id,197,0.11
start_station_name,197,0.11
end_station_id,197,0.11
end_station_name,197,0.11
member_birth_year,8265,4.51
member_gender,8265,4.51


## Step 2: Handle missing values 
- Station info (0.11%) -> drop rows.
- Gender (4.51%) -> fill with 'Unknown' (too much data to drop).
- birth_year (4.51%) -> leave as NaN (numeric column) filling it with a fake number or a text label like 'Umknown' would break age calculations later. 

In [43]:
df_clean = df.copy()
#1-Drop rows with missing station info
before = len(df_clean)
df_clean = df_clean.dropna(subset=['start_station_id', 'end_station_id'])
print(f"Removed {before - len(df_clean)} rows with missing station info.")

#2- Fill missing gender with 'Unknown'
df_clean['member_gender'] = df_clean['member_gender'].fillna('Unknown')

Removed 197 rows with missing station info.


## Step 3: Remove exact duplicate rows

In [33]:
duplicate_count = df_clean.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_count}")

Exact duplicate rows: 4


In [34]:
# Remove exact duplicated rows
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Removed {before - len(df_clean)} duplicate rows")


Removed 4 duplicate rows


## Step 4: Handle outliers — age 
Impossible ages (> 80) are treated as data-entry errors and removed.

In [35]:
#Calculate implied age from birth year (temporary check, not a permenent column yet)
implied_age = 2019 - df_clean['member_birth_year']
print(implied_age.describe())

count    174952.000000
mean         34.196865
std          10.118731
min          18.000000
25%          27.000000
50%          32.000000
75%          39.000000
max         141.000000
Name: member_birth_year, dtype: float64


In [36]:
print((implied_age >80).sum())

192


In [44]:
#Remove rows with impossible age ( > 80)
before = len(df_clean)
implied_age = 2019 - df_clean['member_birth_year']
df_clean = df_clean[(implied_age.isna()) | (implied_age <= 80)]
print(f"Removed {before - len (df_clean)} rows with impossible age (> 80)")

Removed 192 rows with impossible age (> 80)


In [38]:
print(df_clean['duration_sec'].describe())

count    183023.000000
mean        726.218022
std        1795.950646
min          61.000000
25%         325.000000
50%         514.000000
75%         796.000000
max       85444.000000
Name: duration_sec, dtype: float64


In [39]:
# Calculate IQR bounds for duration_sec
q1 = df_clean['duration_sec'].quantile(0.25)
q3 = df_clean['duration_sec'].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

print(f"Upper bound (seconds): {upper_bound}")
print(f"Upper bound (minutes): {upper_bound / 60}")
print(f"Trips above this bound: {(df_clean['duration_sec'] > upper_bound).sum()}")
print(f"Percentage: {(df_clean['duration_sec'] > upper_bound).mean() * 100:.2f}%")

Upper bound (seconds): 1502.5
Upper bound (minutes): 25.041666666666668
Trips above this bound: 10093
Percentage: 5.51%


## Step 5: Handle outliers — trip duration 
- IQR was too aggressive (flags 5.5% of trips at ~25 min).
- Using a domain-based cutoff instead: cap trips longer than 2 hours.

In [40]:
# Check how many trips exceed a 2-hour threshold
outlier_cutoff_sec = 7200  # 2 hours in seconds
print(f"Trips above 2 hours: {(df_clean['duration_sec'] > outlier_cutoff_sec).sum()}")
print(f"Percentage: {(df_clean['duration_sec'] > outlier_cutoff_sec).mean() * 100:.2f}%")

Trips above 2 hours: 711
Percentage: 0.39%


In [41]:
# Cap trip duration at 2 hours instead of dropping the rows
outlier_cutoff_sec = 7200    # 2 hours in seconds
n_capped = (df_clean['duration_sec'] > outlier_cutoff_sec).sum()
df_clean['duration_sec'] = df_clean['duration_sec'].clip(upper=outlier_cutoff_sec)
print(f"Capped {n_capped} trips at {outlier_cutoff_sec} seconds")

Capped 711 trips at 7200 seconds


## Step 6: Export cleaned dataset

In [42]:
df_clean.to_csv('cleaned_fordgobike.csv', index=False)
print("Saved! Shape:", df_clean.shape)

Saved! Shape: (183023, 16)
